# Financial Services Advisor Agent Tutorial

This tutorial uses Microsoft Foundry SDKs to create and test a banking and financial assistance agent. You will:

1. Initialize an `AIProjectClient` with interactive browser authentication.
2. Create a versioned prompt agent for general banking, loan, and investment education.
3. Persist conversation context with the Conversations API.
4. Invoke the agent with the Responses API.
5. Delete only the agent version created during the tutorial.

### Important Financial Disclaimer

> The financial information provided by this notebook is for general educational and demonstration purposes only and is not intended as financial, investment, legal, or tax advice. Always consult qualified professionals before making financial decisions. Past performance does not guarantee future results, and all investments carry risk.

## Authentication Setup

The setup cell uses `InteractiveBrowserCredential`. When you run it, complete the browser sign-in prompt with an identity that can access the Microsoft Foundry project.

Before continuing, ensure the repository `.env` file defines:

- `TENANT_ID`
- `AI_FOUNDRY_PROJECT_ENDPOINT`
- `AZURE_AI_MODEL_DEPLOYMENT_NAME`

## 1. Initial Setup
We'll start by importing needed libraries, loading environment variables, and initializing an **AIProjectClient** so we can do all the agent-related actions. Let's do it! 🎉


In [ ]:
import os
import time

from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition
from azure.identity import InteractiveBrowserCredential
from dotenv import find_dotenv, load_dotenv


dotenv_path = find_dotenv(usecwd=True)
if not dotenv_path:
    raise FileNotFoundError("Could not find a .env file in the current directory or its parents.")

load_dotenv(dotenv_path)

tenant_id = os.getenv("TENANT_ID")
project_endpoint = os.getenv("AI_FOUNDRY_PROJECT_ENDPOINT")
model_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

required_settings = {
    "TENANT_ID": tenant_id,
    "AI_FOUNDRY_PROJECT_ENDPOINT": project_endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": model_name,
}
missing_settings = [name for name, value in required_settings.items() if not value]
if missing_settings:
    raise RuntimeError(f"Missing required environment variables: {', '.join(missing_settings)}")

print("Opening browser-based Azure authentication...")
credential = InteractiveBrowserCredential(tenant_id=tenant_id)
project_client = AIProjectClient(endpoint=project_endpoint, credential=credential)
openai_client = project_client.get_openai_client()
print("Successfully initialized AIProjectClient")

## 2. Creating our Financial Services Advisor Agent 🏦

We'll create an Agent specialized in general banking and financial guidance. We'll explicitly mention disclaimers in its instructions, so it never forgets regulatory compliance! The instructions also ask the agent to focus on banking products, loan inquiries, and general financial guidance while always encouraging users to seek professional advice.

In [ ]:
def create_financial_advisor_agent():
    """Create a versioned financial services advisor agent."""
    agent = project_client.agents.create_version(
        agent_name="financial-services-advisor",
        description="Financial services advisor",
        metadata={"semantic_version": "3.0.1"},
        definition=PromptAgentDefinition(
            model=model_name,
            instructions="""
            You are a friendly AI Financial Services Advisor for a retail bank.
            You provide general information about banking products, loans, and financial services, but always:
            1. Include regulatory and financial disclaimers.
            2. Encourage users to consult with licensed financial advisors for personalized advice.
            3. Provide general, non-personalized guidance around banking, loans, savings, and investments.
            4. Clearly remind users you're not a licensed financial advisor and cannot provide investment recommendations.
            5. Follow responsible banking practices and never suggest risky financial behaviors.
            6. Explain concepts like APR, interest rates, credit scores, and loan terms in simple language.
            """,
        ),
    )
    print(f"Created financial services advisor agent (name: {agent.name}, version: {agent.version})")
    return agent


financial_advisor = create_financial_advisor_agent()

## 3. Managing Financial Service Conversations

A conversation stores messages and model responses across multiple turns. The notebook creates one persisted conversation and reuses its ID for each financial-services question.

In [ ]:
def start_financial_conversation():
    """Create a persisted conversation for banking and financial discussions."""
    conversation = openai_client.conversations.create()
    print(f"Created a new conversation, ID: {conversation.id}")
    return conversation


financial_conversation = start_financial_conversation()

## 4. Asking Banking & Financial Questions 💳
We'll create messages from the user about typical banking questions. For example, **"What factors affect my mortgage interest rate?"** or **"How do I improve my credit score?"**. We'll let our Financial Advisor Agent respond, always remembering the regulatory disclaimers!

In [ ]:
def chat_with_financial_agent(user_question: str, conversation_id: str | None = None) -> str:
    """Send a question to the agent through the Responses API."""
    if not financial_advisor:
        raise RuntimeError("No agent is available. Run the agent creation cell first.")

    print(f"User: {user_question}")
    print("Processing with agent...")

    request_kwargs = {
        "extra_body": {
            "agent_reference": {
                "type": "agent_reference",
                "name": financial_advisor.name,
                "version": financial_advisor.version,
            }
        },
        "input": user_question,
    }
    if conversation_id:
        request_kwargs["conversation"] = conversation_id

    response = openai_client.responses.create(**request_kwargs)
    if not response.output_text:
        raise RuntimeError(f"Response {response.id} did not contain output text.")

    print(f"Agent: {response.output_text}")
    return response.output_text


print("Financial Services Advisor Agent is ready for testing.")

### Example Financial Services Queries
Let's do some quick queries now to see the agent's disclaimers and how it handles typical banking and financial questions. We'll ask about **mortgages**, **credit scores**, and **savings strategies**.

In [ ]:
conversation_id = financial_conversation.id
example_queries = [
    (
        "Mortgage Advice",
        "What factors affect my mortgage interest rate and how can I get a better rate?",
    ),
    (
        "Credit Score Guidance",
        "How is my credit score calculated and what can I do to improve it?",
    ),
    (
        "Savings and Investment Basics",
        "What's the difference between a savings account, CD, and money market account? Which is best for emergency funds?",
    ),
]

responses = []
for test_number, (title, question) in enumerate(example_queries, start=1):
    print(f"\nTest {test_number}: {title}")
    print("-" * 30)
    responses.append(
        chat_with_financial_agent(question, conversation_id=conversation_id)
    )

successful_tests = sum(bool(response) for response in responses)
print(f"\nSummary: {successful_tests}/{len(example_queries)} tests passed successfully")

## 5. Final Test: Complex Financial Question 🧹

In [ ]:
print("Final Test: Complex Financial Scenario")
print("=" * 40)

final_response = chat_with_financial_agent(
    "I'm a 30-year-old first-time homebuyer with $50,000 saved for a down payment. "
    "I have a credit score of 720 and an annual income of $85,000. I'm looking at homes "
    "around $350,000. Should I consider a 15-year or 30-year mortgage, and what other "
    "costs should I budget for when buying a home?",
    conversation_id=financial_conversation.id,
)

print("\nFinal test successful.")

## 6. Cleanup

Delete only the agent version created by this notebook, then close the SDK clients. Other versions with the same agent name are preserved.

In [ ]:
agent_name = financial_advisor.name
agent_version = financial_advisor.version

try:
    print(f"Deleting agent '{agent_name}' version {agent_version}...")
    project_client.agents.delete_version(
        agent_name=agent_name,
        agent_version=agent_version,
    )

    for attempt in range(1, 6):
        remaining_versions = list(project_client.agents.list_versions(agent_name=agent_name))
        created_version_exists = any(
            version.version == agent_version for version in remaining_versions
        )
        if not created_version_exists:
            print("Created agent version deleted successfully.")
            break
        if attempt < 5:
            time.sleep(2)
    else:
        raise RuntimeError(
            f"Agent '{agent_name}' version {agent_version} is still visible after cleanup."
        )
finally:
    openai_client.close()
    project_client.close()
    credential.close()
    print("SDK clients closed.")

# Congratulations! 🏆
You've successfully built a **Financial Services Advisor** that can:
1. **Respond** to basic banking, loan, and financial questions.
2. **Use disclaimers** to ensure regulatory compliance and encourage professional consultation.
3. **Provide** general banking product and savings information.
4. **Use** the synergy of **Microsoft Foundry** modules to power the conversation.

## Next Steps
- Explore adding more advanced tools (like **FileSearchTool** or **CodeInterpreterTool**) to analyze loan documents or financial data.
- Evaluate your AI's performance with **azure-ai-evaluation**!
- Add **OpenTelemetry** or Azure Monitor for deeper insights.
- Incorporate **function calling** if you want to handle things like loan calculations or account lookups.

Happy banking automation! 💰

## Summary

You have now:

- Connected to a Microsoft Foundry project with interactive browser authentication.
- Created a versioned prompt agent with `PromptAgentDefinition`.
- Created a persisted conversation with the Conversations API.
- Invoked the agent through the Responses API using an agent reference.
- Reused one conversation for multi-turn context.
- Deleted only the agent version created by this notebook.

### Key Concepts

1. **Agent versions** define a named agent's model and instructions.
2. **Conversations** persist context across Responses API calls.
3. **Responses** invoke the referenced agent and return generated output.
4. **Cleanup scope** matters when multiple versions share one agent name.